# Disaster Tweets：从基线到 Kaggle 提交的完整记录

**任务**：Kaggle [Natural Language Processing with Disaster Tweets](https://www.kaggle.com/competitions/nlp-getting-started) —— 二分类，判断推文是否描述真实灾难。
**指标**：F1（正类 = disaster）。
**数据**：train **7613** / test **3263**（正类比例 0.4297）。

**本文档记录**：实验协议、每个实验的 CV 与 LB、提交流程与踩坑、下一步计划。
所有实验都在本地 RTX 3050 Laptop (4GB) 上完成。


## 1. 结果一览（含 Kaggle 榜位）

| ID | 实验 | 模型 | CV F1 | ±std | 阈值 | **public LB** | 备注 |
|---|---|---|---|---|---|---|---|
| **EXP-0006** | **blend_v1** | BERT + TF-IDF 融合 | 0.8089 | — | 0.46 | **0.83849** | ✅ 已提交（rank 70/442） |
| EXP-0004 | blend_oof_probe | BERT + TFIDF | 0.8110 | — | 0.41 | — | OOF 融合探针（仅分析） |
| EXP-0003 | bert_base_v1 | bert-base-uncased | 0.8085 | 0.0100 | 0.44 | — | 微调 5 折，3 epoch |
| EXP-0005 | bert_base_v2 | bert-base-uncased | 0.8071 | 0.0083 | 0.54 | — | 5 epoch（**未提升，3 epoch 已饱和**） |
| EXP-0002 | tfidf_lr_v1 | TF-IDF + LR | 0.7699 | 0.0156 | 0.41 | — | 基线（word 1-2 + char_wb 3-5, C=1） |
| EXP-0001 | tfidf_lr_v0_wordonly | TF-IDF + LR | 0.7617 | 0.0093 | — | — | 无字符 n-gram |

### 榜位分析（2026-10-07，共 442 队）

| 项 | 值 |
|---|---|
| 我们的名次 | **第 70 / 442** |
| 分数 ≥0.99 的队伍 | 7 支（**几乎必然是使用了 test 标签泄漏**） |
| 剔除疑似作弊后名次 | **第 63 / 435** |
| 合法榜第一 | 0.85412 |
| **与合法第一的差距** | **0.0156** |

### 结论

1. **微调预训练模型是主要收益来源**：相对 TF-IDF 基线 **+0.0386 CV**
2. 融合（BERT 0.551 + TFIDF 0.449）再 +0.0018
3. **CV(0.8089) 与 LB(0.83849) 存在系统性偏差 +0.0296** —— CV 适合做相对排序，绝对值不可直接当 LB 估计
4. epochs 3→5 无收益 → 该数据规模下 BERT-base 已饱和；后续增益要靠**更强骨干 + 集成**


In [ ]:
# 读取实验记录表（本地有 experiments.csv 时直接读；在 Kaggle 上则用内置数据）
import pandas as pd
try:
    df = pd.read_csv("experiments.csv").sort_values("f1_mean", ascending=False)
except FileNotFoundError:
    df = pd.DataFrame([
        ("EXP-0006","blend_v1","blend",0.8089,None,0.46,"0.83849"),
        ("EXP-0004","blend_oof_probe","blend",0.8110,None,0.41,""),
        ("EXP-0003","bert_base_v1","bert-base-uncased",0.8085,0.0100,0.44,""),
        ("EXP-0005","bert_base_v2","bert-base-uncased",0.8071,0.0083,0.54,""),
        ("EXP-0002","tfidf_lr_v1","tfidf_lr",0.7699,0.0156,0.41,""),
        ("EXP-0001","tfidf_lr_v0_wordonly","tfidf_lr",0.7617,0.0093,None,""),
    ], columns=["exp_id","name","model","f1_mean","f1_std","threshold","lb_score"])
cols = [c for c in ["exp_id","name","model","f1_mean","f1_std","threshold","lb_score"] if c in df.columns]
df[cols]


## 2. 评估协议（先立规矩，再调模型）

1. **固定 5 折 `StratifiedKFold(shuffle=True, random_state=42)`**，分层依据可选：
   - `target`（常规）—— 本项目基准 `target5-seed42`
   - `keyword`（复现赛题真实切分：每个 keyword 同时出现在 train/test，CV 更贴 LB）
2. **指标用 F1**，输出每折值与 均值±std；**绝不在训练集上评估**
3. **每次实验保存 OOF 预测** → 阈值搜索与模型融合都依赖它
4. **阈值在 OOF 上搜**，不写死 0.5
5. **不为 public LB 反复试探**；新实验只有 CV F1 ≥ 当前最佳才保留


## 3. 预处理

- 文本构造：`keyword + " " + 轻清洗(text)`
  - `html.unescape`（`&amp;` → `&`）
  - URL → `" url "`、`@xxx` → `" user "`（**保留"有没有链接/提及"的信息**）
  - 空白归一
- **刻意不做**：词干化 / 去停用词 / 去标点
  > Transformer 依赖自然语言形态与上下文；这条路线做词干化会明显掉分（TF-IDF 路线则可以）
- 分词：HF `AutoTokenizer(use_fast=True)`，`truncation=True, max_length=128, padding=False`
- **动态 padding**：`DataCollatorWithPadding`，每个 batch 补到 batch 内最长


## 4. 训练技术

| 技术 | 设置 | 作用 |
|---|---|---|
| AdamW 参数分组 | bias / LayerNorm.weight 的 weight_decay=**0.0**，其余 0.01 | 标准做法 |
| 学习率 | **2e-5** | BERT 微调甜点区 |
| 调度 | **线性 warmup 10% + 线性衰减** | 防早期大梯度破坏预训练权重 |
| 混合精度 | fp16 (`torch.amp.autocast` + `GradScaler`) | 4GB 显存的关键 |
| 梯度累积 | `loss/accum`，总步数按累积后计算 | 等效大 batch、省显存 |
| 梯度裁剪 | `clip_grad_norm_(1.0)`，**在 `scaler.unscale_` 之后** | 顺序错了会裁到缩放后的梯度（AMP 常见坑） |
| 早停 | 盯 **val F1**，patience 2，**restore_best_weights** | 对齐竞赛指标；避免用最后一轮权重 |
| 交叉验证集成 | 各折模型概率平均 | 免费稳定性提升 |
| 后处理 | OOF 上网格搜阈值 | 二分类 F1 常有小幅收益 |
| 模型融合 | BERT 0.551 + TF-IDF 0.449（权重在 OOF 上搜） | +0.0018 |


## 5. 环境与权重获取（两个真实的坑）

**坑 1：HuggingFace 被墙** → 用 **ModelScope** 拉权重。
脚本内置回退链：`本地路径 → models/ 缓存 → modelscope.snapshot_download → HF id`。
已验证可用：`google-bert/bert-base-uncased`、`microsoft/deberta-v3-base`、`AI-ModelScope/distilbert-base-uncased`、`AI-ModelScope/roberta-base`。

**坑 2：Kaggle 提交要 PUT 到 `www.googleapis.com`（Google 云存储），该域被墙** → 提交会一直卡住直到超时。
解决办法（三步式提交拆开走）：

```
第 1 步 创建上传会话  → 本机直连 kaggle.com（不受影响）
第 2 步 上传文件      → 交给【Windows 侧 curl.exe + 系统代理】完成
第 3 步 创建提交      → 本机直连 kaggle.com
```

对应脚本 `submit_via_proxy.py`：自动从 Windows 注册表读代理端口、自动把文件放到 Windows 可见路径、代理不通时秒级报错。


In [ ]:
# 复现命令清单（在项目目录执行）
cmds = [
 "python tfidf_lr_baseline.py --log --name tfidf_lr_v1 "
 "--changes 'TF-IDF(word1-2+char_wb3-5) + LR C=1; keyword拼接; 轻清洗'",

 "python finetune_transformer.py --model google-bert/bert-base-uncased "
 "--folds 5 --epochs 3 --bs 16 --name bert_base_v1 --log --changes '微调BERT-base'",

 "python finetune_transformer.py --model microsoft/deberta-v3-base "
 "--folds 5 --epochs 4 --bs 8 --accum 2 --name deberta_v1 --log --changes '微调DeBERTa-v3-base'",

 "python blend.py --names bert_base_v2 tfidf_lr_v1 --out submissions/blend_v1.csv "
 "--log --exp-name blend_v1",

 "python submit_via_proxy.py --file submissions/blend_v1.csv",

 "python exp_log.py list",
]
print("\n".join(cmds))


## 6. 下一步计划（按性价比）

1. **DeBERTa-v3-base 5 折**（预期 CV +0.005~0.01）
2. **多种子平均**（`--seeds 3`）
3. **三方融合**（BERT + DeBERTa + TF-IDF）后提交
4. **LLRD（层级学习率衰减）**、R-Drop：成本低、通常小幅增益
5. 外部数据（DFE `disasters-on-social-media`）：**丢弃 `Can't Decide`** + 与 train/test **按归一化文本去重**
6. 伪标签：高置信 test 回训一轮

**当前与合法榜第一仅差 0.0156**，上述 1~3 项叠加有望把 LB 推到 0.845+。


## 7. 红线（不要做的事）

- ❌ 使用 test 标签泄漏（DFE 数据集按 `id` 可直接还原 test 标签）—— 榜单上 7 支 ≥0.99 的队伍几乎都是这么来的，分数没有意义
- ❌ 在训练集上评估模型（in-sample 比较会让过拟合模型"看起来最好"）
- ❌ 用 lr=1e-3 微调 BERT；给 BERT 喂词干化/去停用词文本
- ❌ 验证集建在带 shuffle 的流上（每 epoch 变化、与训练集重叠）
- ❌ 训练完直接用最后一轮权重预测（应用最优 checkpoint）
- ❌ 靠 public LB 反复试探挑模型
